# Обучение и проверка (Colab, GPU)

Тонкий блокнот: весь код в пакете `anemia/` и в `scripts/` репозитория. Запускайте ячейки по порядку.

Архив CP-AnemiC (`CP-AnemiC_dataset.zip`, общая папка проекта `md/history/097_2026-10-02_dataset_cp_anemic/`):
положите в `MyDrive/anemia/` на Диске или перетащите в панель «Файлы» слева; иначе ячейка 2 попросит загрузить его.
Данные и модели хранятся на Диске в `MyDrive/anemia/` (без Диска — в `/content/anemia`), в репозиторий не попадают.

In [ ]:
#@title Ячейка 1 — репозиторий, зависимости, хранилище
#@markdown Google Диск сохраняет данные и модели между сессиями. Если Диск не подключается — снимите галочку:
#@markdown всё будет в /content (стирается при закрытии сессии, данные скачаются заново).
Использовать_Google_Диск = False  #@param {type:"boolean"}
import os, subprocess, sys
REPO = 'zhansaya-afk/jjghg'
BRANCH = 'main'
BASE = '/content/anemia'
if Использовать_Google_Диск:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        BASE = '/content/drive/MyDrive/anemia'
    except Exception as e:
        print('⚠️ Google Диск не подключился, работаю без него (данные в /content/anemia):', str(e)[:150])
os.makedirs(BASE, exist_ok=True)
os.environ['ANEMIA_BASE'] = BASE
os.environ['ANEMIA_DATA'] = f'{BASE}/data'
os.environ['ANEMIA_MODELS'] = f'{BASE}/models'

# токен нужен, только если репозиторий приватный: Colab → ключ слева → Secrets → GITHUB_TOKEN
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.isdir('/content/repo/.git'):
    subprocess.run(['git', '-C', '/content/repo', 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/repo', 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/repo'], check=True)
os.chdir('/content/repo')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data,dev]'], check=True)
print('Хранилище:', BASE)
print('Коммит:', subprocess.run(['git', 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title Ячейка 2 — данные: скачать (один раз) и собрать manifest.csv
import os, glob, shutil
D, BASE = os.environ['ANEMIA_DATA'], os.environ['ANEMIA_BASE']
if not os.path.exists(f'{D}/eyes_defy_anemia/labels.csv'):
    !python scripts/fetch_data.py --eyes-defy
if not os.path.exists(f'{D}/cp_anemic/labels.csv'):
    # архив CP-AnemiC ищется на Диске (MyDrive/anemia/) или в /content (панель «Файлы» слева);
    # если его нигде нет — появится кнопка загрузки с компьютера
    found = sorted(glob.glob(f'{BASE}/*AnemiC*.zip') + glob.glob('/content/*AnemiC*.zip'))
    if not found:
        try:
            from google.colab import files
            print('Загрузите CP-AnemiC_dataset.zip (5 МБ):')
            up = files.upload()
            found = [os.path.abspath(n) for n in up if n.lower().endswith('.zip')]
        except Exception as e:
            print('⚠️ загрузка не удалась:', str(e)[:120])
    if found:
        cp_zip = found[0]
        !python scripts/fetch_data.py --cp-zip "{cp_zip}"
    else:
        print('⚠️ CP-AnemiC пропущен: нет архива')
!python scripts/prepare_data.py
!python -m pytest -q

In [ ]:
#@title Ячейка 3 — честный baseline (этап 1): сегментация v11 + Hb по цвету, все датасеты
# ≈10–20 мин на CPU/GPU Colab. Пришлите Claude весь вывод ячейки: таблицу и строки CSV в конце.
import os, pandas as pd
before = len(pd.read_csv('results/metrics_history.csv')) if os.path.exists('results/metrics_history.csv') else 0
!python scripts/evaluate.py --tag baseline_v11
h = pd.read_csv('results/metrics_history.csv')
new = h.iloc[before:]
new.to_csv(f"{os.environ['ANEMIA_BASE']}/metrics_history_colab.csv", mode='a', index=False,
           header=not os.path.exists(f"{os.environ['ANEMIA_BASE']}/metrics_history_colab.csv"))
print('\n--- строки для results/metrics_history.csv ---')
print(new.to_csv(index=False))